In [1]:
# Célula 1 (revisada) — DuckDB + MinIO com Secrets Manager
import duckdb

con = duckdb.connect("../data/curso_analytics.duckdb")
con.execute("INSTALL httpfs; LOAD httpfs;")

# Segredo TEMPORÁRIO: vale para esta conexão quando feita entre containers Docker. Rode esta célula sempre que reabrir o notebook.
con.execute("""
    CREATE OR REPLACE SECRET minio (
        TYPE s3,
        KEY_ID 'minioadmin',
        SECRET 'minioadmin123',
        ENDPOINT 'host.docker.internal:9000',
        URL_STYLE 'path',
        USE_SSL false,
        REGION 'us-east-1'
    );
""")

# Segredo Temporário usando localhost: vale para as conexões feitas pelo host. Rode esta célula apenas uma vez.
#con.execute("""
#    CREATE OR REPLACE SECRET minio (
#        TYPE s3,
#        KEY_ID 'minioadmin',
#        SECRET 'minioadmin123',
#        ENDPOINT 'localhost:9000',
#        URL_STYLE 'path',
#        USE_SSL false,
#        REGION 'us-east-1'
#    );
#""")

print(con.sql("SELECT name, type, persistent FROM duckdb_secrets()"))

┌─────────┬─────────┬────────────┐
│  name   │  type   │ persistent │
│ varchar │ varchar │  boolean   │
├─────────┼─────────┼────────────┤
│ minio   │ s3      │ false      │
└─────────┴─────────┴────────────┘



In [2]:
# Célula 2 (revisada) — Catálogo Iceberg (SQLite local + MinIO)
import io, os, time
from datetime import datetime, timezone

import boto3
import pandas as pd
import pyarrow as pa
from pyiceberg.catalog.sql import SqlCatalog

os.makedirs("../data/iceberg_catalog", exist_ok=True)

MINIO = {
    "s3.endpoint": "http://host.docker.internal:9000",
    "s3.access-key-id": "minioadmin",
    "s3.secret-access-key": "minioadmin123",
    "s3.region": "us-east-1",            # evita a tentativa de descobrir a região do bucket
}

#MINIO = {
#    "s3.endpoint": "http://localhost:9000",
#    "s3.access-key-id": "minioadmin",
#    "s3.secret-access-key": "minioadmin123",
#    "s3.region": "us-east-1",            # evita a tentativa de descobrir a região do bucket
#}


catalog = SqlCatalog(
    "curso_lakehouse",
    **{"uri": "sqlite:///../data/iceberg_catalog/catalog.db", "warehouse": "s3://silver", **MINIO},
)
catalog.create_namespace_if_not_exists("bronze_processado")
print("Namespaces:", catalog.list_namespaces())


Namespaces: [('bronze_processado',)]


In [3]:
# Célula 3 — Schema fortemente tipado + particionamento oculto (hidden partitioning)
from pyiceberg.schema import Schema
from pyiceberg.types import (NestedField, LongType, IntegerType, StringType,
                             DoubleType, TimestampType, BooleanType)
from pyiceberg.partitioning import PartitionSpec, PartitionField
from pyiceberg.transforms import MonthTransform

schema_pedidos = Schema(
    NestedField(1,  "id_pedido",         LongType(),      required=True),
    NestedField(2,  "id_cliente",        LongType(),      required=True),
    NestedField(3,  "id_produto",        LongType(),      required=True),
    NestedField(4,  "categoria",         StringType()),
    NestedField(5,  "valor_unitario",    DoubleType()),
    NestedField(6,  "quantidade",        IntegerType()),
    NestedField(7,  "valor_total",       DoubleType()),
    NestedField(8,  "status_pedido",     StringType()),
    NestedField(9,  "regiao",            StringType()),
    NestedField(10, "data_pedido",       TimestampType()),
    NestedField(11, "avaliacao_cliente", IntegerType()),
    NestedField(12, "_fonte",            StringType()),
    NestedField(13, "_ingerido_em",      StringType()),
    identifier_field_ids=[1],            # chave lógica: usada pelo upsert
)

# O particionamento é declarado sobre uma TRANSFORMAÇÃO da coluna (mês de data_pedido).
# Quem consulta filtra por data_pedido e não precisa conhecer a estrutura de pastas.
spec_particao = PartitionSpec(
    PartitionField(source_id=10, field_id=1000, transform=MonthTransform(), name="data_pedido_mes")
)

NOME_TABELA = "bronze_processado.pedidos_ecommerce"
tabela = catalog.create_table_if_not_exists(
    identifier=NOME_TABELA,
    schema=schema_pedidos,
    partition_spec=spec_particao,
    properties={
        "write.parquet.compression-codec": "zstd",
        "write.metadata.compression-codec": "gzip",
    },
)
print(tabela.schema())
print(tabela.spec())
print("Localização:", tabela.location())

table {
  1: id_pedido: required long
  2: id_cliente: required long
  3: id_produto: required long
  4: categoria: optional string
  5: valor_unitario: optional double
  6: quantidade: optional int
  7: valor_total: optional double
  8: status_pedido: optional string
  9: regiao: optional string
  10: data_pedido: optional timestamp
  11: avaliacao_cliente: optional int
  12: _fonte: optional string
  13: _ingerido_em: optional string
}
[
  1000: data_pedido_mes: month(10)
]
Localização: s3://silver/bronze_processado/pedidos_ecommerce


In [4]:
# Célula 4 — Leitura da Bronze e primeira carga
s3 = boto3.client(
    "s3",
    endpoint_url=MINIO["s3.endpoint"],          # http://host.docker.internal:9000 (definido na Célula 2)
    aws_access_key_id=MINIO["s3.access-key-id"],
    aws_secret_access_key=MINIO["s3.secret-access-key"],
    region_name="us-east-1",
)

arquivos = [
    o["Key"] for o in s3.list_objects_v2(Bucket="bronze", Prefix="ecommerce_sintetico/").get("Contents", [])
    if o["Key"].endswith(".parquet")
]
df_bronze = pd.read_parquet(io.BytesIO(s3.get_object(Bucket="bronze", Key=arquivos[0])["Body"].read()))
print(f"Bronze: {len(df_bronze):,} linhas em {arquivos[0]}")

COLUNAS = [campo.name for campo in tabela.schema().fields]

def preparar_lote(df: pd.DataFrame) -> pa.Table:
    """Converte um DataFrame da Bronze para o schema Arrow da tabela Iceberg."""
    df = df.copy()
    df["avaliacao_cliente"] = pd.to_numeric(df["avaliacao_cliente"], errors="coerce").astype("Int32")
    df["quantidade"] = df["quantidade"].astype("int32")
    return pa.Table.from_pandas(df[COLUNAS], schema=tabela.schema().as_arrow(), preserve_index=False)

inicio = time.time()
tabela.append(preparar_lote(df_bronze.iloc[:50_000]))
print(f"✅ Snapshot 1: 50.000 registros em {time.time() - inicio:.2f}s")

Bronze: 1,000,000 linhas em ecommerce_sintetico/ano=2026/mes=10/dia=08/pedidos.parquet
✅ Snapshot 1: 50.000 registros em 2.12s


In [5]:
# Célula 5 — Segunda carga (Snapshot 2) e histórico de snapshots
tabela.append(preparar_lote(df_bronze.iloc[50_000:100_000]))
tabela = catalog.load_table(NOME_TABELA)

def listar_snapshots(t):
    for s in sorted(t.snapshots(), key=lambda s: s.timestamp_ms):
        ts = datetime.fromtimestamp(s.timestamp_ms / 1000, tz=timezone.utc)
        print(f"{s.snapshot_id}  {ts:%H:%M:%S}  {s.summary.operation.value:<9}"
              f" +{s.summary.get('added-records', '0'):>6}  total={s.summary.get('total-records')}")

listar_snapshots(tabela)

# Alternativa tabular (metadados como tabela Arrow)
tabela.inspect.snapshots().to_pandas()[["committed_at", "snapshot_id", "operation"]]

1033362292451318245  22:41:17  append    + 50000  total=50000
4931607074258359137  22:42:56  append    + 50000  total=100000


,committed_at,snapshot_id,operation
0,2026-11-05 22:41:17.501,1033362292451318245,append
1,2026-11-05 22:42:56.133,4931607074258359137,append


In [6]:
# Célula 6 — Time travel por ID e por instante
snapshots = sorted(tabela.snapshots(), key=lambda s: s.timestamp_ms)
v1 = snapshots[0]                                     # o mais antigo é o PRIMEIRO em ordem cronológica

n_v1    = tabela.scan(snapshot_id=v1.snapshot_id).to_arrow().num_rows
n_atual = tabela.scan().to_arrow().num_rows
print(f"Snapshot inicial: {n_v1:,} | Atual: {n_atual:,}")

# "Como estava a tabela neste instante?"
s = tabela.snapshot_as_of_timestamp(v1.timestamp_ms)
print("Snapshot vigente naquele instante:", s.snapshot_id)

# Leitura com filtro e projeção: o Iceberg poda partições e arquivos pelas estatísticas dos manifests
df = tabela.scan(
    row_filter="status_pedido == 'cancelado' and data_pedido >= '2022-01-15T00:00:00'",
    selected_fields=("id_pedido", "valor_total", "data_pedido"),
).to_pandas()
print(f"Cancelados após 15/01: {len(df):,}")

Snapshot inicial: 50,000 | Atual: 100,000
Snapshot vigente naquele instante: 1033362292451318245
Cancelados após 15/01: 5,996


In [7]:
# Célula 7 — Direito de eliminação (LGPD art. 18): apagar todos os pedidos de um cliente
cliente = int(tabela.scan(selected_fields=("id_cliente",), limit=1).to_pandas().iloc[0, 0])
print("Antes :", tabela.scan(row_filter=f"id_cliente == {cliente}").to_arrow().num_rows)

tabela.delete(delete_filter=f"id_cliente == {cliente}")
tabela = catalog.load_table(NOME_TABELA)

print("Depois:", tabela.scan(row_filter=f"id_cliente == {cliente}").to_arrow().num_rows)
print("Operação:", tabela.current_snapshot().summary.operation)

Antes : 2
Depois: 0
Operação: Operation.OVERWRITE


In [ ]:
# Célula 8 — UPSERT (MERGE) pela chave id_pedido
lote = tabela.scan(row_filter="id_pedido <= 3").to_pandas()
lote["status_pedido"] = "reembolsado"                 # 3 pedidos alterados

novo = lote.iloc[[0]].copy()
novo["id_pedido"], novo["status_pedido"] = 2_000_001, "pendente"   # 1 pedido novo

resultado = tabela.upsert(
    pa.Table.from_pandas(pd.concat([lote, novo]), schema=tabela.schema().as_arrow(), preserve_index=False),
    join_cols=["id_pedido"],
)
print(f"Atualizados: {resultado.rows_updated} | Inseridos: {resultado.rows_inserted}")
tabela = catalog.load_table(NOME_TABELA)
listar_snapshots(tabela)